# Deep learning on images

## Load modules from repo

In [1]:
# Le code suivant dans un notebook permet de :
# - autoriser les imports de fichiers python de ce repo
# - spécifier les chemins relativement à la racine du repo plutôt que relativement au notebook

import os

# Ce code cherche le dossier racine en remontant dans l'arborescence
# jusqu'à ce qu'il trouve le dossier 'src'.
# Cela le rend indépendant de l'endroit où vous lancez le notebook.
try:
    # On part du dossier du notebook
    notebook_dir = os.path.dirname(__file__)
except NameError:
    # __file__ n'existe pas en mode interactif, on utilise le répertoire de travail
    notebook_dir = os.getcwd()

# On remonte jusqu'à trouver un dossier contenant 'src'
project_root = notebook_dir
while not os.path.isdir(os.path.join(project_root, 'src')):
    parent_dir = os.path.dirname(project_root)
    if parent_dir == project_root: # On a atteint la racine du système
        raise FileNotFoundError("Impossible de trouver le dossier 'src'. Vérifiez la structure du projet.")
    project_root = parent_dir

os.chdir(project_root)

In [2]:
os.getcwd()

'/home/val/Documents/Dev/DataScientest/Rakuten'

In [3]:
import src
from src.preprocessing.core import load_reproducible_split
from src.preprocessing.pipelines.deep_learning_on_images import preprocess_features, load_preprocessors, save_preprocessors
from src.models.on_images.deep_learning import define_model
from src.models.on_text_and_images.deep_learning import get_custom_hyperparams, log_experiment

from pathlib import Path
import pandas as pd
# import numpy as np

2025-10-06 11:20:55.367623: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-06 11:20:55.846893: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-10-06 11:20:57.234076: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1759742458.616205    6610 gpu_device.cc:2020] Created device /job:localhost/rep

In [4]:
import importlib
importlib.reload(src.preprocessing.core)
importlib.reload(src.preprocessing.pipelines.deep_learning_on_images)
importlib.reload(src.models.on_images.deep_learning)
importlib.reload(src.models.on_text_and_images.deep_learning)

<module 'src.models.on_text_and_images.deep_learning' from '/home/val/Documents/Dev/DataScientest/Rakuten/src/models/on_text_and_images/deep_learning.py'>

## Load tensorflow

In [5]:
import tensorflow as tf

In [6]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))

Num GPUs Available:  1


In [7]:
print(f"tensorflow: {tf.__version__}")

tensorflow: 2.20.0


## Load split dataset

In [8]:
X_train, X_test, y_train, y_test = load_reproducible_split(folder = 'Dataset2')
full_y_train=y_train

In [9]:
# Distribution of class proportions
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024767
min       0.008994
25%       0.018268
50%       0.031458
75%       0.056137
max       0.120223
Name: proportion, dtype: float64

## Configuration 1

In [10]:
version=1
artifacts_folder=Path(f'artifacts/on_images/deep_learning/v{version}')
log_file_path=artifacts_folder / 'experiments.parquet'

# fit_preprocessors=True
fit_preprocessors=False

small_train_sample = True  # TODO: set to False for real training or fitting preprocessors
frac=0.1

rebalance_with_weights = True  # True is useful if some classes are ignored by the model. In that case, small_train_sample should be False.

augment=True  # Augment data for training

BATCH_SIZE = 32

RANDOM_SEED = 42

# load_model=True
load_model=False

## Preprocessing

In [11]:
if fit_preprocessors and small_train_sample:
    raise ValueError("When fit_preprocessors=True, small_train_sample should be False so that encoders are fitted on the full training dataset.")

In [12]:
if rebalance_with_weights:
    print('using class weights')
else:
    print('not using class weights')

using class weights


In [13]:
if small_train_sample:
    print(f"{small_train_sample=}")
    X_train=X_train.sample(frac=frac, random_state=RANDOM_SEED)
    y_train=y_train.loc[X_train.index]

small_train_sample=True


In [14]:
y_train.value_counts(normalize=True).describe()

count    27.000000
mean      0.037037
std       0.024700
min       0.008980
25%       0.018696
50%       0.031797
75%       0.054541
max       0.122185
Name: proportion, dtype: float64

In [15]:
y_train.value_counts().describe()

count     27.000000
mean     251.592593
std      167.786325
min       61.000000
25%      127.000000
50%      216.000000
75%      370.500000
max      830.000000
Name: count, dtype: float64

In [16]:
print(X_train.shape)

(6793, 31)


In [17]:
import joblib
if fit_preprocessors:
    preprocessors = {}
    print('will fit')
else: # load preprocessors
    print('loading')
    preprocessors = load_preprocessors(names=['target','tabular','hash'],artifacts_folder=artifacts_folder)

loading
artifacts/on_images/deep_learning/v1/preprocessors/target.joblib
artifacts/on_images/deep_learning/v1/preprocessors/tabular.joblib
artifacts/on_images/deep_learning/v1/preprocessors/hash.joblib


In [18]:
train_ds, new_preprocessors, class_weights, train_inputs_dict, y_train_ohe = preprocess_features(X_train, y_train, preprocessors, full_y_train=full_y_train, shuffle=True, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=rebalance_with_weights, augment=augment)
preprocessors |= new_preprocessors
n_cols_tabular = train_inputs_dict['tabular_input'].shape[1]
n_cols_tabular

24

In [19]:
new_preprocessors

{}

In [20]:
# Save on disk
if fit_preprocessors:
    print('Saving.')
    save_preprocessors(new_preprocessors,artifacts_folder)

In [21]:
test_ds, new_preprocessors, class_weights_test, test_inputs_dict, y_test_ohe = preprocess_features(X_test, y_test, preprocessors, shuffle=False, BATCH_SIZE = BATCH_SIZE, rebalance_with_weights=False, augment=False)

## Model

In [22]:
from tensorflow import keras

### Load or create model

In [23]:
import pandas as pd

try:
    # Load tracker
    tracking_df = pd.read_parquet(log_file_path)
    if tracking_df.empty:
        last_experiment = {}
    else:
        last_experiment = tracking_df.iloc[-1].to_dict()
except FileNotFoundError:
    last_experiment = {}

champion_path=None
if load_model:
    # Récupérer le total d'epochs déjà entraînées
    total_epochs_trained = last_experiment.get('total_epochs', 0)

    subversion = last_experiment.get('subversion', 1)

    # Charger le meilleur modèle connu jusqu'à présent (le "champion")
    champion_path = last_experiment.get('best_model_path', None)
    if champion_path and Path(champion_path).exists():
        print(f"Reprise depuis le meilleur modèle de l'expérience : {champion_path}")
        model = keras.models.load_model(champion_path)
    else:
        raise FileNotFoundError("Consider setting load_model=False.")
else:
    subversion = last_experiment.get('subversion', 0)
    total_epochs_trained = 0
    last_experiment = {}
    print("Création d'un nouveau modèle.")
    model, base_model = define_model(pHash_vocab_size=len(preprocessors['hash'].categories_[0]), md5_vocab_size=len(preprocessors['hash'].categories_[1]), n_cols_tabular=n_cols_tabular, num_classes=27)


Création d'un nouveau modèle.


In [24]:
if not load_model:
    subversion = int(input(f"subversion (architecture)? (last: {subversion})"))

In [25]:
subversion

9

### Summary

In [26]:
# model.summary()

## Callbacks

### ModelCheckpoint

In [27]:
# # Pick an available filename to save a model.
# subversion=1
# while True:
#     new_location_for_saving_model = Path(artifacts_folder / f'best_model-{subversion}.keras')
#     if not new_location_for_saving_model.exists():
#         break
#     subversion+=1
# # new_location_for_saving_model_weights = Path(artifacts_folder / f'best_model-{subversion}.h5')
# new_location_for_saving_model


In [28]:
new_location_for_saving_model = artifacts_folder / "candidate_best_model.keras"

In [29]:
# Callback pour sauvegarder le meilleur modèle au fur et à mesure
save = keras.callbacks.ModelCheckpoint(
    new_location_for_saving_model,
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [30]:
# # ou sauvegarder l'intégralité du modèle après l'entraînement, c'est-à-dire son architecture et ses poids :
# model.save(new_location_for_saving_model)

### EarlyStopping

In [31]:
from tensorflow.keras.callbacks import EarlyStopping

# Ce callback arrêtera l'entraînement si la métrique ne s'améliore pas
# pendant n epochs consécutives. Il restaurera aussi les poids du meilleur epoch.
early_stopping = EarlyStopping(
    monitor='val_loss', # La métrique à surveiller
    patience=5,             # Nombre d'epochs à attendre sans amélioration
    mode='min',             # Choisir le mode selon la métrique à surveiller
    restore_best_weights=True # À la fin, le modèle aura les poids de son meilleur score
)

### ReduceLROnPlateau

In [32]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss', # On surveille la perte sur la validation
    factor=0.2,         # On réduit le LR
    patience=2,         # Si métrique stagne pendant n epochs
    min_lr=1e-6,        # On ne descend pas en dessous de cette valeur
    verbose=1           # Affiche un message quand le LR est réduit
)

### TerminateOnNaN

In [33]:
import tensorflow.keras.callbacks as callbacks
terminate_on_nan = callbacks.TerminateOnNaN()

### TensorBoard

In [34]:
import datetime
tensor_board_folder = artifacts_folder / "tensorboard_logs"
timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensor_board = callbacks.TensorBoard(
    log_dir = tensor_board_folder / timestamp,
    histogram_freq=1 # Demande à Keras de logger les distributions des poids et biais à chaque epoch
)

## Configuration 2

### max_epochs

In [35]:
import math
typical_minutes_per_epoch=2.058333 * X_train.shape[0] / 6793

In [ ]:
# max_epochs=13

# # Calculate expected duration
# available_minutes=math.ceil(max_epochs * typical_minutes_per_epoch)
# available_minutes

In [42]:
# Pick max_epochs based on your available time
available_minutes=30

max_epochs=math.floor(available_minutes/typical_minutes_per_epoch)
max_epochs

14

### compilation and callbacks

In [43]:
learning_rate=0.001

In [44]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [45]:
callbacks = [save, early_stopping, reduce_lr, terminate_on_nan, tensor_board]

## Training

In [46]:
raise Exception(f"Are you sure you want to launch training with {available_minutes=}, {max_epochs=}, {champion_path=} ?")

Exception: Are you sure you want to launch training with available_minutes=30, max_epochs=14, champion_path=None ?

In [47]:
import time
start = time.time()
model_history = model.fit(train_ds, validation_data=test_ds, epochs=total_epochs_trained + max_epochs, initial_epoch=max(total_epochs_trained-1,0), callbacks=callbacks)
end = time.time()
total_minutes = (end - start)/60
f"{total_minutes=}"

Epoch 1/14


2025-10-06 11:21:50.580285: I external/local_xla/xla/service/service.cc:163] XLA service 0x74e694011d70 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-10-06 11:21:50.580318: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2025-10-06 11:21:50.943649: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-06 11:21:52.300429: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91300
2025-10-06 11:21:53.273237: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 11:21:53.

212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 175ms/step - accuracy: 0.2533 - loss: 3.1918

2025-10-06 11:22:58.312106: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:22:58.406541: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:22:59.056866: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:22:59.156821: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:22:59.835254: E external/local_xla/xla/stream_

213/213 ━━━━━━━━━━━━━━━━━━━━ 0s 248ms/step - accuracy: 0.2537 - loss: 3.1900

2025-10-06 11:24:16.470926: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2025-10-06 11:24:24.182577: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:24:24.280341: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-10-06 11:24:25.090473: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please invest

213/213 ━━━━━━━━━━━━━━━━━━━━ 169s 651ms/step - accuracy: 0.3395 - loss: 2.8201 - val_accuracy: 0.4910 - val_loss: 2.2111 - learning_rate: 0.0010
Epoch 2/14
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 479ms/step - accuracy: 0.4798 - loss: 2.1858 - val_accuracy: 0.5256 - val_loss: 2.0046 - learning_rate: 0.0010
Epoch 3/14
213/213 ━━━━━━━━━━━━━━━━━━━━ 103s 486ms/step - accuracy: 0.5329 - loss: 1.9738 - val_accuracy: 0.5535 - val_loss: 1.8954 - learning_rate: 0.0010
Epoch 4/14
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 481ms/step - accuracy: 0.5853 - loss: 1.7965 - val_accuracy: 0.5658 - val_loss: 1.8902 - learning_rate: 0.0010
Epoch 5/14
213/213 ━━━━━━━━━━━━━━━━━━━━ 103s 483ms/step - accuracy: 0.6735 - loss: 1.5802 - val_accuracy: 0.5697 - val_loss: 1.9301 - learning_rate: 0.0010
Epoch 6/14
212/213 ━━━━━━━━━━━━━━━━━━━━ 0s 180ms/step - accuracy: 0.7574 - loss: 1.3645
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.00020000000949949026.
213/213 ━━━━━━━━━━━━━━━━━━━━ 102s 481ms/step - accuracy: 0.7892 - 

'total_minutes=16.43843602736791'

## Evaluation

In [48]:
print(f"source venv/bin/activate\ntensorboard --logdir '{tensor_board_folder.resolve()}'")
# To use tensorboard:
# open a terminal and run the two commands below.

source venv/bin/activate
tensorboard --logdir '/home/val/Documents/Dev/DataScientest/Rakuten/artifacts/on_images/deep_learning/v1/tensorboard_logs'


In [49]:
tracker = {}
tracker['X_train.shape[0]']=X_train.shape[0]
tracker['actual_epochs']=model_history.epoch[-1] + 1
tracker['minutes_per_epoch'] = total_minutes / tracker['actual_epochs']
tracker

{'X_train.shape[0]': 6793,
 'actual_epochs': 9,
 'minutes_per_epoch': 1.8264928919297678}

In [50]:
tracker['minutes_per_epoch']=max(tracker['minutes_per_epoch'], last_experiment.get('minutes_per_epoch',0))

In [51]:
#Takes 1m40
y_pred = model.predict(test_ds)
y_pred

531/531 ━━━━━━━━━━━━━━━━━━━━ 73s 126ms/step


array([[2.34328021e-04, 3.26263579e-03, 1.05342884e-02, ...,
        1.11956811e-02, 2.53484992e-04, 9.80291516e-04],
       [1.10924931e-03, 5.17540006e-03, 8.89218692e-03, ...,
        2.84779649e-02, 9.19942453e-04, 5.92893315e-03],
       [1.03607730e-04, 9.87581350e-03, 2.72529013e-02, ...,
        2.10930541e-01, 1.44086051e-04, 1.15927099e-03],
       ...,
       [2.78926373e-01, 1.87450834e-02, 9.81760677e-04, ...,
        1.98614944e-04, 3.65306914e-01, 3.17966240e-03],
       [1.08843145e-03, 5.72344009e-03, 3.39631597e-03, ...,
        1.53508885e-02, 7.24769547e-04, 1.08857977e-03],
       [2.42654942e-02, 2.73783039e-02, 8.43658447e-02, ...,
        5.98467067e-02, 5.62637998e-03, 1.19843218e-03]],
      shape=(16984, 27), dtype=float32)

In [52]:
from sklearn import metrics

In [53]:
y_test_class = preprocessors['target'].inverse_transform(y_test_ohe.argmax(axis=1))
y_pred_class = preprocessors['target'].inverse_transform(y_pred.argmax(axis=1))

In [54]:
pd.set_option('display.max_columns',None)
pd.crosstab(y_test_class, y_pred_class, rownames=['True'], colnames=['Predicted'])

Predicted,10,40,50,60,1140,1160,1180,1280,1281,1300,1301,1302,1320,1560,1920,1940,2060,2220,2280,2403,2462,2522,2582,2583,2585,2705,2905
True,,,,,,,,,,,,,,,,,,,,,,,,,,,
10,303,5,0,0,9,56,1,4,0,1,0,0,3,0,5,1,4,0,73,64,2,50,0,1,0,41,0
40,22,161,13,6,12,96,0,7,0,27,0,3,2,7,3,1,8,0,31,17,12,15,3,9,6,40,1
50,1,4,110,20,21,20,0,7,2,53,0,6,7,6,0,0,11,0,1,10,11,21,4,21,0,0,0
60,0,7,10,114,1,8,0,1,1,8,0,0,0,0,0,0,1,0,2,1,5,4,0,1,0,0,2
1140,3,11,5,1,377,39,0,35,2,8,0,1,8,0,1,1,7,0,6,7,1,7,0,11,0,1,2
1160,4,8,1,0,13,729,0,1,0,1,0,0,0,0,0,0,0,0,14,14,4,2,0,0,0,0,0
1180,2,1,0,0,52,30,0,10,2,4,0,0,6,1,0,0,7,0,10,12,0,8,0,5,0,3,0
1280,2,8,17,2,123,18,0,346,8,189,1,19,25,23,13,3,62,1,5,9,3,38,5,40,9,2,3
1281,10,18,7,2,19,60,0,67,34,19,0,13,7,5,0,7,22,0,13,16,4,50,2,18,2,10,9


In [55]:
candidate_f1_score=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
champion_f1_score = last_experiment.get('weighted_avg_f1_score', 0.0)
candidate_f1_score, champion_f1_score

(0.5398482620755286, 0.0)

In [56]:
tracker['weighted_avg_f1_score'] = candidate_f1_score

In [57]:
import pandas as pd
report=metrics.classification_report(y_test_class, y_pred_class, output_dict=True)
report=pd.DataFrame(report).transpose()
report

,precision,recall,f1-score,support
10,0.450893,0.486356,0.467954,623.000000
40,0.559028,0.320717,0.407595,502.000000
50,0.320700,0.327381,0.324006,336.000000
60,0.690909,0.686747,0.688822,166.000000
1140,0.488342,0.705993,0.577335,534.000000
1160,0.560338,0.921618,0.696941,791.000000
1180,0.000000,0.000000,0.000000,153.000000
1280,0.419394,0.355236,0.384658,974.000000
1281,0.586207,0.082126,0.144068,414.000000
1300,0.556470,0.737364,0.634271,1009.000000


In [58]:
# Summary over classes
report.iloc[:-3].describe()

,precision,recall,f1-score,support
count,27.000000,27.000000,27.000000,27.000000
mean,0.552541,0.479490,0.476040,629.037037
std,0.184740,0.271236,0.222237,420.759339
min,0.000000,0.000000,0.000000,153.000000
25%,0.448504,0.313833,0.356714,310.000000
50%,0.560338,0.486356,0.486339,534.000000
75%,0.671696,0.697686,0.653912,953.500000
max,0.897436,0.921618,0.829132,2042.000000


In [59]:
# negative correlation between support and another measure would suggest class imbalance hurts performance
report.corr()

,precision,recall,f1-score,support
precision,1.000000,0.386650,0.497322,0.004123
recall,0.386650,1.000000,0.968960,0.084505
f1-score,0.497322,0.968960,1.000000,0.074498
support,0.004123,0.084505,0.074498,1.000000


In [60]:
tracker['weighted_avg_f1_score']=metrics.f1_score(y_test_class, y_pred_class, average='weighted')
tracker['weighted_avg_f1_score']

0.5398482620755286

In [61]:
tracker['min_f1_score']=report['f1-score'].iloc[:-3].describe()['min']
tracker['std_f1_score']=report['f1-score'].iloc[:-3].describe()['std']
tracker

{'X_train.shape[0]': 6793,
 'actual_epochs': 9,
 'minutes_per_epoch': 1.8264928919297678,
 'weighted_avg_f1_score': 0.5398482620755286,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.22223735039316334)}

## Update tracker

In [71]:
tracker['comment']="Disabled first dropout layer."
tracker['comment']

'Disabled first dropout layer.'

In [63]:
import numpy as np
if candidate_f1_score > champion_f1_score:
    print("Nouveau champion ! Sauvegarde du modèle.")
    keep_candidate=True

    best_epoch_in_session_idx = np.argmax(model_history.history['val_accuracy'])
    best_epoch_global = total_epochs_trained + best_epoch_in_session_idx
    best_val_accuracy = model_history.history['val_accuracy'][best_epoch_in_session_idx]
    tracker['val_accuracy'] = best_val_accuracy

    # Construire le nouveau nom de fichier informatif
    new_champion_filename = f"best_model_sv-{subversion}_epoch_index-{best_epoch_global:02d}_val_accuracy-{best_val_accuracy:.4f}_f1-{candidate_f1_score:.4f}.keras"
    new_champion_path = artifacts_folder / new_champion_filename

    # Renommer le fichier candidat pour en faire le nouveau champion
    os.rename(new_location_for_saving_model, new_champion_path)

    tracker['best_model_path'] = str(new_champion_path)
    print(new_champion_filename)

    # Supprimer l'ancien champion pour garder le dossier propre
    if champion_path and Path(champion_path).exists():
        print("Effacement de l'ancien modèle de la même subversion {champion_path=} .")
        os.remove(champion_path)
else:
    print("Le candidat n'a pas battu le champion.")
    keep_candidate=False


Nouveau champion ! Sauvegarde du modèle.
best_model_sv-9_epoch_index-06_val_accuracy-0.5755_f1-0.5398.keras


In [64]:
tracker['epoch_index'] = best_epoch_global
tracker['total_epochs'] = total_epochs_trained + best_epoch_global + 1

In [65]:
to_track=['subversion','max_epochs','rebalance_with_weights','BATCH_SIZE','learning_rate','timestamp']
tracker |= {k: globals()[k] for k in to_track}
tracker |= get_custom_hyperparams(model, base_model)

In [72]:
tracker

{'X_train.shape[0]': 6793,
 'actual_epochs': 9,
 'minutes_per_epoch': 1.8264928919297678,
 'weighted_avg_f1_score': 0.5398482620755286,
 'min_f1_score': np.float64(0.0),
 'std_f1_score': np.float64(0.22223735039316334),
 'comment': 'Disabled first dropout layer.',
 'val_accuracy': 0.5755416750907898,
 'best_model_path': 'artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-06_val_accuracy-0.5755_f1-0.5398.keras',
 'epoch_index': np.int64(6),
 'total_epochs': np.int64(7),
 'subversion': 9,
 'max_epochs': 14,
 'rebalance_with_weights': True,
 'BATCH_SIZE': 32,
 'learning_rate': 0.001,
 'timestamp': '20251006-112105',
 'dense_layers_sizes': {'tabular_dense_1': 64,
  'image_dense': 128,
  'tabular_dense_2': 32,
  'final_dense_1': 256},
 'embedding_dims': {'pHash_embedding': 16, 'md5_embedding': 16}}

In [67]:
if not keep_candidate:
    raise Exception("Sure you want to delete current candidate model?")

In [68]:
if not keep_candidate:
    os.remove(new_location_for_saving_model)

In [69]:
raise Exception("Sure you want to save the current values of `tracker`? Consider editing the comment.")

Exception: Sure you want to save the current values of `tracker`? Consider editing the comment.

In [73]:
# Saves tracker as a new row of parquet file (creates the parquet file if necessary)
log_experiment(tracker, loaded_model=load_model, log_file_path=log_file_path)

Log pour l'expérience subversion 9 mis à jour dans artifacts/on_images/deep_learning/v1/experiments.parquet .


## Show tracking logs

In [74]:
pd.set_option('max_colwidth', None)

In [75]:
tracking_df = pd.read_parquet(log_file_path)
tracking_df

,subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,total_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,timestamp,comment,best_model_path
0,2,False,6793,32,1.998601,10,0.001,0.569713,0.528593,0.0,0.247884,256_128_64_32,16_16,None,Decreased frac from 0.3 to 0.1 for small train sample size to try reducing overfitting even on a small sample. Added data augmentation. Changed Dropout rate from .5 to .7. Added L2 penalties to dense layers.,artifacts/on_images/deep_learning/v1/best_model_sv-2_epochs-10_f1-0.5286.keras
1,2,False,20380,32,3.867331,8,0.001,0.584609,0.563841,0.0,0.239801,256_128_64_32,16_16,None,Increased frac from 0.1 to 0.3.,artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-02_val_loss-1.7413_f1-0.5638.keras
2,3,False,6793,32,2.024143,10,0.001,0.566710,0.527659,0.0,0.243439,128_64_32_16,8_8,None,"Decreased frac from 0.3 to 0.1. Set dropout rate from 0.7 to 0.5. Reduced dense layers and hash embeddings. Training curve is less steep and validation curve goes higher, which suggests overfitting has been reduced.",artifacts/on_images/deep_learning/v1/best_model_sv-1_epoch_index-04_val_loss-1.8077_f1-0.5277.keras
3,4,True,6793,32,1.954997,13,0.001,0.541922,0.490818,0.0,0.270646,128_64_32_16,8_8,None,"Added Dropout(0.5) after concatenation layer. Set rebalance_with_weights=True. Less overfitting: performance is better on validation than on train until epoch 7, and train performance curve is less steep.",artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-07_val_loss-1.8789_f1-0.4908.keras
4,4,True,20380,32,3.115025,4,0.001,0.562529,0.499731,0.0,0.276383,128_64_32_16,8_8,None,Increased frac from .1 to .3.,artifacts/on_images/deep_learning/v1/best_model_sv-4_epoch_index-03_val_loss-1.7963_f1-0.4997.keras
5,5,True,20380,32,3.080263,3,0.001,0.566945,0.506920,0.0,0.255305,128_64_32_16,8_8,None,Removed dropout layer before softmax. Performance better but more overfitting.,artifacts/on_images/deep_learning/v1/best_model_sv-5_epoch_index-02_val_loss-1.7414_f1-0.5069.keras
6,6,True,6793,32,1.781595,6,0.001,0.558585,0.537182,0.0,0.223538,256_128_64_32,16_16,None,Frac back to .1. Reverted layers/embeddings to higher sizes and uncommented Dropout before softmax.,artifacts/on_images/deep_learning/v1/best_model_sv-6_epoch_index-05_val_loss-1.9153_f1-0.5372.keras
7,7,True,6793,16,2.911669,5,0.001,0.473976,0.410351,0.0,0.261518,256_128_64_32,16_16,20251004-151611,Unfreezed base model. Batch size from 32 to 16 because memory error.,artifacts/on_images/deep_learning/v1/best_model_sv-7_epoch_index-04_val_loss-2.1782_f1-0.4104.keras
8,8,True,6793,32,1.819877,4,0.001,0.573775,0.523869,0.0,0.235614,256_128_64_32,16_16,20251006-101001,"Unfreezed base model, so batch size back to 32. Lowered first dropout rate from .5 to .2.",artifacts/on_images/deep_learning/v1/best_model_sv-8_epoch_index-03_val_loss-1.9175_f1-0.5239.keras
9,9,True,6793,32,1.826493,7,0.001,0.575542,0.539848,0.0,0.222237,256_128_64_32,16_16,20251006-112105,Disabled first dropout layer.,artifacts/on_images/deep_learning/v1/best_model_sv-9_epoch_index-06_val_accuracy-0.5755_f1-0.5398.keras


In [ ]:
old_log_file_path='artifacts/on_images/deep_learning/v1/2025-10-02 experiments-v1.parquet'

In [ ]:
tracking_df0 = pd.read_parquet(old_log_file_path)
tracking_df0

,subversion,started_from_subversion,rebalance_with_weights,X_train.shape[0],BATCH_SIZE,minutes_per_epoch,max_epochs,actual_epochs,learning_rate,val_accuracy,weighted_avg_f1_score,min_f1_score,std_f1_score,dense_layers_sizes,embedding_dims,comment
0,1,<NA>,False,6793,32,2.058333,2,2,0.001,0.538100,0.498881,0.000000,0.238552,256_128_64_32,16_16,"Test dataset was shuffled, which induces bias on the metric. Seed not set on small training sample. No callbacks."
1,2,1,False,6793,32,1.807693,4,4,0.001,0.595737,0.585677,0.179894,0.192174,256_128_64_32,16_16,Test dataset no longer shuffled. Seed set on small training sample. Callbacks set.
2,3,1,False,6793,32,1.674025,29,7,0.001,0.599976,0.593786,0.182796,0.191047,256_128_64_32,16_16,
3,4,3,False,20380,32,2.657786,8,4,0.001,0.608220,0.598100,0.254545,0.180580,256_128_64_32,16_16,Increased frac from 0.1 to 0.3 for small train sample size.
